# JobFit RAG Assistant

Colab-compatible notebook for `python -m src.jobfit.function_caller`.

**RAM fix**: The original `vector_store.py` loads `all-MiniLM-L6-v2` at import time.
This notebook lazy-loads it so both embedding model and LLM are never in RAM at once.

In [ ]:
# Cell 1 — Install dependencies
# !pip install transformers torch sentence-transformers chromadb scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 90.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 123.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 28.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0

In [2]:
# Cell 2 — vector_store.py  (lazy embedder to avoid OOM)
import chromadb
from sentence_transformers import SentenceTransformer

_embedder = None

def _get_embedder():
    global _embedder
    if _embedder is None:
        _embedder = SentenceTransformer("all-MiniLM-L6-v2")
    return _embedder

client = chromadb.PersistentClient(path="chroma_db")

resume_collection = client.get_or_create_collection("resume")
jd_collection = client.get_or_create_collection("job_description")

def add_to_collection(collection, chunks: list[str], doc_type: str):
    embedder = _get_embedder()
    embeddings = embedder.encode(chunks).tolist()
    ids = [f"{doc_type}_{i}" for i in range(len(chunks))]
    metadatas = [{"type": doc_type} for _ in chunks]
    collection.upsert(
        documents=chunks,
        ids=ids,
        metadatas=metadatas,
        embeddings=embeddings
    )

def retrieve(query: str, collection, k: int = 5) -> list[str]:
    embedder = _get_embedder()
    query_embedding = embedder.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=k
    )
    return results['documents'][0]

In [ ]:
# Cell 3 — similarity.py  (already lazy-loaded)
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

SIMILARITY_THRESHOLD = 0.50

_embedding_model = None

def _get_embedding_model():
    global _embedding_model
    if _embedding_model is None:
        _embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
    return _embedding_model

def compute_similarity(resume_sentences, jd_sentences):
    embedding_model = _get_embedding_model()
    resume_embeddings = embedding_model.encode(resume_sentences)
    jd_embeddings = embedding_model.encode(jd_sentences)
    similarity_matrix = cosine_similarity(jd_embeddings, resume_embeddings)
    return similarity_matrix

def get_best_matches(resume_sentences, jd_sentences):
    matches = []
    similarity_matrix = compute_similarity(resume_sentences, jd_sentences)
    for i, jd in enumerate(jd_sentences):
        best_index = similarity_matrix[i].argmax()
        best_score = similarity_matrix[i][best_index]
        best_resume = resume_sentences[best_index]
        matches.append({
            "jd": jd,
            "resume": best_resume,
            "score": float(best_score)
        })
    return matches

In [ ]:
# Cell 4 — validator.py
import re

def validate_json(output, expected_schema):
    for key, expected_type in expected_schema.items():
        if key not in output:
            return False
        if expected_type and not isinstance(output[key], expected_type):
            return False
    return True

In [ ]:
# Cell 5 — llm.py
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

_tokenizer = None
_model = None
_device = None

def _load_model():
    global _tokenizer, _model, _device
    if _model is None:
        _device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Loading {MODEL_NAME} on {_device}...")
        _tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        _model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True
        ).to(_device)
        _model.eval()
        print("Model ready.")
    return _tokenizer, _model, _device

def generate_text(prompt, system_prompt=None, temperature=0.7, top_k=50, top_p=0.9, max_new_tokens=200):
    tokenizer, model, device = _load_model()
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=True,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = outputs[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [6]:


def calculate_match_score(query: str = "skills experience projects"):
    resume_chunks = retrieve(query, resume_collection, k=3)
    jd_chunks = retrieve(query, jd_collection, k=3)
    matches = get_best_matches(resume_chunks, jd_chunks)
    avg_score = sum(m["score"] for m in matches) / len(matches) if matches else 0
    top = [m for m in matches if m["score"] >= SIMILARITY_THRESHOLD]
    return {"score": round(avg_score, 3), "top_matches": top}

def extract_skills(query: str = "technical skills experience tools"):
    resume_chunks = retrieve(query, resume_collection, k=3)
    text = " ".join(resume_chunks)
    prompt = (
        "Extract all technical and professional skills from the text below. "
        "Return them as a comma-separated list. No explanation.\n\n"
        f"Text:\n{text}\n\nSkills:"
    )
    result = generate_text(
        prompt,
        system_prompt="You are a skill extractor. Return only a comma-separated list. No explanation. No extra text.",
        temperature=0.2,
        max_new_tokens=100
    )
    skills = [s.strip().strip("[]\"'") for s in result.replace("\n", ",").split(",") if s.strip()]
    return skills if skills else []

def summarize_jd_llm(query: str = "role responsibilities requirements skills"):
    jd_chunks = retrieve(query, jd_collection, k=3)
    jd_text = " ".join(jd_chunks)
    prompt = (
        "Summarize this job description in 2-3 sentences. "
        "Focus on role, key responsibilities, and required skills.\n\n"
        f"Job Description:\n{jd_text}\n\nSummary:"
    )
    return generate_text(
        prompt,
        system_prompt="You are a job description summarizer. Be concise and factual. No extra commentary.",
        temperature=0.3,
        max_new_tokens=150
    )

In [9]:
# Cell 3 — Populate ChromaDB with sample data
resume_chunks = [
    "Experienced software engineer skilled in Python, C++, and data structures",
    "Built and deployed ML models using scikit-learn and PyTorch",
    "Used Docker and FastAPI to containerize and serve REST APIs",
    "Worked with MLflow for experiment tracking and model versioning",
    "Implemented CI/CD pipelines using GitHub Actions"
]

jd_chunks = [
    "We are looking for an AI Engineer with experience in Python and Machine Learning",
    "Candidate must know Docker, FastAPI and MLOps tools",
    "Experience with model deployment and monitoring is required",
    "Strong understanding of CI/CD pipelines and version control",
    "Familiarity with experiment tracking tools like MLflow is a plus"
]

add_to_collection(resume_collection, resume_chunks, "resume")
add_to_collection(jd_collection, jd_chunks, "jd")

print("✅ ChromaDB populated")

✅ ChromaDB populated


In [7]:
# Cell 7 — function_caller.py  (unchanged logic)
import json

TOOL_SCHEMAS = [
    {
        "name": "calculate_match_score",
        "description": "Calculates how well a resume matches a job description. Use when user asks about fit, match, or compatibility.",
        "parameters": {
            "type": "object",
            "properties": {
                "resume_sentences": {"type": "array", "description": "List of resume sentences"},
                "jd_sentences": {"type": "array", "description": "List of JD sentences"}
            },
            "required": ["resume_sentences", "jd_sentences"]
        }
    },
    {
        "name": "summarize_jd",
        "description": "Summarizes a job description into key points. Use when user wants a quick understanding of a JD.",
        "parameters": {
            "type": "object",
            "properties": {
                "jd_text": {"type": "string", "description": "Raw job description text"}
            },
            "required": ["jd_text"]
        }
    },
    {
        "name": "extract_skills",
        "description": "Extracts skills from any text (resume or JD). Use when user asks what skills are present or required.",
        "parameters": {
            "type": "object",
            "properties": {
                "text": {"type": "string", "description": "Text to extract skills from"}
            },
            "required": ["text"]
        }
    }
]

ROUTING_SCHEMAS = [
    {"name": schema["name"], "description": schema["description"]}
    for schema in TOOL_SCHEMAS
]

TOOL_REGISTRY = {
    "calculate_match_score": calculate_match_score,
    "summarize_jd": summarize_jd_llm,
    "extract_skills": extract_skills,
}

TOOL_CALL_SCHEMA = {"name": str}
RESULT_SCHEMAS = {
    "calculate_match_score": {"score": float, "top_matches": list},
}

def parse_tool_call(response: str):
    cleaned = re.sub(r"```json|```", "", response).strip()
    match = re.search(r"\{.*?\}", cleaned, re.DOTALL)
    if not match:
        return None
    try:
        parsed = json.loads(match.group())
        if "name" in parsed and parsed["name"] in TOOL_REGISTRY:
            return {"name": parsed["name"]}
        return None
    except json.JSONDecodeError:
        return None

_KEYWORD_MAP = {
    "calculate_match_score": {"match", "compare", "fit", "compatible"},
    "summarize_jd": {"summarize", "summary", "overview", "summarise"},
    "extract_skills": {"skill", "extract"},
}

def _keyword_fallback(query: str):
    q = query.lower()
    for tool_name, keywords in _KEYWORD_MAP.items():
        if any(kw in q for kw in keywords):
            return tool_name
    return None

def run_tool_workflow(user_query: str, verbose: bool = True):
    if verbose:
        print(f"\n{'='*50}")
        print(f"USER: {user_query}")
        print(f"{'='*50}")
    routing_context = json.dumps(ROUTING_SCHEMAS, indent=2)
    decision_prompt = f"""You have access to these tools:
{routing_context}

RULES:
- If a tool matches the user's intent, respond ONLY with JSON in this exact format:
{{"name": "tool_name"}}
- Replace tool_name with one of: calculate_match_score, summarize_jd, extract_skills
- If no tool matches, reply in plain text directly.
- No explanation. No markdown. No extra keys.

User: {user_query}
"""
    llm_output = generate_text(decision_prompt, temperature=0.1, max_new_tokens=50)
    if verbose:
        print(f"\n[LLM Raw] {llm_output}")
    tool_call = parse_tool_call(llm_output)
    routing = "llm"
    if tool_call and not validate_json(tool_call, TOOL_CALL_SCHEMA):
        if verbose:
            print("[Validate] Malformed tool call — falling back")
        tool_call = None
    if not tool_call:
        tool_name = _keyword_fallback(user_query)
        if tool_name:
            routing = "keyword"
            if verbose:
                print(f"[Fallback] Keyword matched → {tool_name}")
        else:
            direct_answer = generate_text(user_query, temperature=0.3, max_new_tokens=200)
            return {"answer": direct_answer, "tool": None, "routing": "direct"}
    else:
        tool_name = tool_call["name"]
    if verbose:
        print(f"[Tool] {tool_name} | Route: {routing}")
    if tool_name not in TOOL_REGISTRY:
        return {"answer": f"Unknown tool: {tool_name}", "tool": tool_name, "routing": routing}
    try:
        result = TOOL_REGISTRY[tool_name]()
        if verbose:
            print(f"[Result] {result}")
    except Exception as e:
        return {"answer": f"Tool error: {e}", "tool": tool_name, "routing": routing}
    result_schema = RESULT_SCHEMAS.get(tool_name)
    if result_schema and isinstance(result, dict) and not validate_json(result, result_schema):
        return {"answer": "Tool returned data in wrong format.", "tool": tool_name, "routing": routing}
    if isinstance(result, str):
        final_answer = result
    else:
        final_prompt = f"""The user asked: "{user_query}"
Tool used: {tool_name}
Result: {result}

Give a clear, short final answer based on the result."""
        final_answer = generate_text(final_prompt, temperature=0.3, max_new_tokens=150)
    if verbose:
        print(f"\n[Final] {final_answer}")
    return {"answer": final_answer, "tool": tool_name, "routing": routing}

In [8]:
# Cell 8 — Run the tests
tests = [
    "How well does my resume match this job?",
    "Summarize this job description for me",
    
    "What skills are in this resume?",
]
for q in tests:
    out = run_tool_workflow(q, verbose=True)


USER: How well does my resume match this job?
Loading HuggingFaceTB/SmolLM2-1.7B-Instruct on cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

...

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


USER: Summarize this job description for me

[LLM Raw] {"name": "summarize_jd"}
[Tool] summarize_jd | Route: llm
[Result] Job Title: Marketing Manager

Responsibilities:
- Develop and implement marketing strategies to promote products and services
- Manage and coordinate marketing campaigns across various channels
- Analyze market trends and competitor activity to inform marketing decisions
- Collaborate with cross-functional teams to ensure alignment and effective execution of marketing plans
- Oversee budget and resource allocation for marketing initiatives

Required Skills:
- Bachelor's degree in Marketing or related field
- 3+ years of experience in marketing management
- Strong understanding of marketing principles and strategies
- Excellent analytical and problem-solving skills
- Excellent communication and leadership skills

[Final] Job Title: Marketing Manager

Responsibilities:
- Develop and implement marketing strategies to promote products and services
- Manage and coordina

In [10]:
# Cell 3 — Test Data + Populate ChromaDB

# ── Raw inputs ────────────────────────────────────────────────────────────────
resume_text = """Experienced software engineer skilled in Python, C++ and data structures.
Built and deployed ML models using scikit-learn and PyTorch.
Used Docker and FastAPI to containerize and serve REST APIs.
Worked with MLflow for experiment tracking and model versioning.
Implemented CI/CD pipelines using GitHub Actions."""

jd_text = """We are looking for an AI Engineer with experience in Python and Machine Learning.
Candidate must know Docker, FastAPI and MLOps tools.
Experience with model deployment and monitoring is required.
Strong understanding of CI/CD pipelines and version control.
Familiarity with experiment tracking tools like MLflow is a plus."""

# ── Split into chunks ─────────────────────────────────────────────────────────
resume_chunks = [s.strip() for s in resume_text.split(".") if s.strip()]
jd_chunks     = [s.strip() for s in jd_text.split(".") if s.strip()]

print(f"Resume chunks: {len(resume_chunks)}")
print(f"JD chunks:     {len(jd_chunks)}")

# ── Store in ChromaDB ─────────────────────────────────────────────────────────
add_to_collection(resume_collection, resume_chunks, "resume")
add_to_collection(jd_collection, jd_chunks, "jd")

print("✅ ChromaDB populated\n")

# ── Run tests ─────────────────────────────────────────────────────────────────
tests = [
    "How well does my resume match this job?",
    "Summarize this job description for me",
    "What skills are in this resume?",
]

for q in tests:
    out = run_tool_workflow(q, verbose=True)

Resume chunks: 5
JD chunks:     5
✅ ChromaDB populated


USER: How well does my resume match this job?

[LLM Raw] {"name": "calculate_match_score"}
[Tool] calculate_match_score | Route: llm
[Result] {'score': 0.579, 'top_matches': [{'jd': 'We are looking for an AI Engineer with experience in Python and Machine Learning', 'resume': 'Experienced software engineer skilled in Python, C++ and data structures', 'score': 0.5948969721794128}, {'jd': 'Familiarity with experiment tracking tools like MLflow is a plus', 'resume': 'Worked with MLflow for experiment tracking and model versioning', 'score': 0.8150909543037415}]}

[Final] Your resume matches the job description to a moderate extent.

USER: Summarize this job description for me

[LLM Raw] {"name": "summarize_jd"}
[Tool] summarize_jd | Route: llm
[Result] The job requires experience with model deployment and monitoring, proficiency in Docker, FastAPI, and MLOps tools, and knowledge of CI/CD pipelines and version control.

[Final] The jo